In [ ]:
import os
import os.path as op

import importlib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpecFromSubplotSpec

# import nibabel as nib
# from nilearn import datasets, image
# from matplotlib.colors import LinearSegmentedColormap
# from scipy.optimize import linear_sum_assignment
# from scipy.spatial.distance import cdist
# from scipy.stats import spearmanr
# from seaborn import kdeplot
# import networkx as nx
# from time import perf_counter

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

from joblib import Parallel, delayed
from tqdm.notebook import tqdm


import sys
sys.path.append("../..")
path_to_data = op.join("../..", "data")

from bimodularity import dgsp
from bimodularity import bimod_plots as plot
from bimodularity import palettes
from bimodularity import data_load as dload
from bimodularity import bundle as b_utils

In [ ]:
cmaps = plot.get_all_cmaps()
print(cmaps.keys())

In [ ]:
path_to_brain_data = op.join("../../data", "brain")
path_to_bundles = op.join(path_to_brain_data, "SCIL")

path_to_derivatives = op.join("../../data", "brain", "derivatives")

path_to_sc = op.join(path_to_derivatives, "structural_connectome")
path_to_ec = op.join(path_to_derivatives, "atlas_correspondence", "rDCM")
path_to_lobe = op.join(path_to_derivatives, "atlas_correspondence", "Lobes")
path_to_consensus = op.join(path_to_derivatives, "consensus_clustering")

fast = False

In [ ]:
all_scales = [1, 2, 3, 4]

use_delay = False
k_threshold = 0
b_thresh = 0
slines_theshold = 5

struct_type = ""

scale = 2

k_matrix, labels, node_centers = dload.load_bundle_graph(path_to_data=path_to_sc,
                                                         data_suffix="Laus2018_",
                                                         scale=scale,
                                                         b_prob_threshold=0,
                                                         slines_theshold=slines_theshold,
                                                         log_slines=False,
                                                         normalize_slines=False,
                                                         verbose=True)

_, node_centers = b_utils.fix_thalamus(labels, pos=node_centers)
labels, k_matrix = b_utils.fix_thalamus(labels, matrix=k_matrix)
order_by_lobe, lobe_sizes, lobe_labels, lobe_df = dload.get_lobe_info(scale, labels, path_to_lobe=path_to_lobe)

## Consensus Clustering

In [ ]:
scale = 2
n_vec_max = 20
gamma = 1

nosquared = False

n_trials = 50
n_init = 50
dthresh = 0
selected_sort = 0

upsample = 3

all_n_kmeans = np.arange(10, 80, 1)

bm_fname = "_".join([
    f"brain_consensus-EC",
    f"scale{scale}",
    f"nvec{n_vec_max}" + "-NoSquared"*nosquared,
    f"trials{n_trials}",
    f"ninit{n_init}",
    f"kmeans{all_n_kmeans[0]}-{all_n_kmeans[-1]}",
    f"slines_thresh{slines_theshold}",
    f"ustruct{struct_type}"*(struct_type != ""),
    f"dthresh{dthresh}"*(dthresh != 0),
    f"gamma{gamma}"*(gamma != 1),
    f"selected{selected_sort}"*(selected_sort != 0)
    ])
bm_fname += ".pkl"

if op.isfile(op.join(path_to_consensus, bm_fname)):
    data = dload.load(op.join(path_to_consensus, bm_fname))
    graph = data["graph"]
    all_n_kmeans = data["all_n_kmeans"]
    cons_lab = data["cons_lab"]
    avg_cons = data["avg_cons"]
    edge_to_bundle_mat = data["edge_to_bundle_mat"]
else:
    print("No benchmark file found - Please, run the benchmarking notebook!")
    print(f"\t`{bm_fname}`")

fig_suffix = f"gamma{gamma}_nvec{n_vec_max}" + "-NoSquared"*nosquared
path_to_figures = op.join("./supplementary", f"scale{scale}", fig_suffix)

os.makedirs(path_to_figures, exist_ok=True)

In [ ]:
centroid_dir = op.join(
    path_to_brain_data, "BundleAtlas", "centroids",
    f"scale{scale}", f"group_centroids_scale{scale}"
)

a_bundles_labels = pd.read_csv(op.join(path_to_bundles, "bundle_names.csv"))
a_bundles_labels = a_bundles_labels.rename(columns={"Unnamed: 0": "BundleNum"})
a_bundles_labels.loc[:, "BundleNameShort"] = [lab.replace("_Brainstem", "_Bstm") for lab in a_bundles_labels.loc[:, "BundleName"]]

if "edge_to_bundle_Bmdf" in data:
    edge_to_bundle_dist = data["edge_to_bundle_Bmdf"]
else:
    edge_to_bundle_dist = b_utils.compute_edge_to_bundle_distance(
        graph, labels=labels, scale=scale,
        path_to_edge_centroids=centroid_dir,
        path_to_atlas_centroids=path_to_bundles, a_bundles_labels=a_bundles_labels
    )

    data.update({"edge_to_bundle_Bmdf": edge_to_bundle_dist})
    dload.save(op.join(path_to_consensus, bm_fname), data)

## SI-DirectedConnectome

In [ ]:
importlib.reload(plot)
# plot.plot_kde()

fig, all_axes = plt.subplots(ncols=2, figsize=(12, 6), gridspec_kw={"width_ratios": [1, 1], "wspace": 0.3})

all_axes[0].imshow(graph[order_by_lobe][:, order_by_lobe], cmap=cmaps["div_rb"], interpolation="None")
plot.plot_lobe_lines(all_axes[0], lobe_sizes, lobe_labels, fontsize=16)


all_axes[1].axis("off")
gs = GridSpecFromSubplotSpec(3, 1, subplot_spec=all_axes[1].get_subplotspec(),
                             height_ratios=[0.5, 6, 1], hspace=0)
axes = [fig.add_subplot(gs[1])]*2

nzer_mask = np.logical_or(graph > 0, (graph == 1).T)
plot.plot_kde(graph[nzer_mask], ax=axes[1], data_range=np.linspace(0, 1, 100),
              bw_adjust=0.1, fill=True, color=cmaps["div_rb"](0.8))

med = np.median(graph[nzer_mask])
perc25 = np.percentile(graph[nzer_mask], 25)
perc75 = np.percentile(graph[nzer_mask], 75)

iszer = np.logical_and(graph < 0.01, k_matrix > 0)
iszer = np.logical_and(graph.T != 0, iszer)
isone = graph > 0.99 

print("N Edges", (graph > 0).sum())

med_text = f"{(np.abs(graph - 0.5) < 1e-2).sum():d} edges are close to 0.5\n$(0.49 < a_{{ij}} < 0.51)$"
axes[1].text(0.8, 0.95, med_text, ha="center", va="top")

dir_text = f"{iszer.sum():d} edges are close to 1\n({100*iszer.sum()/(graph > 0).sum():1.3f}%)"
axes[1].text(0.8, 0.5, dir_text, ha="center", va="top")

axes[1].plot([med]*2, [0, 1], color="k", lw=2)
axes[1].plot([perc25]*2, [0, 0.6], color="k", ls=":", lw=2)
axes[1].plot([perc75]*2, [0, 0.6], color="k", ls=":", lw=2)

plot.plot_kde(graph[nzer_mask], ax=axes[1], data_range=np.linspace(0, 1, 100),
              bw_adjust=0.1, fill=False, color=cmaps["div_rb"](0.8))

axes[1].text(med, 1, f"Median\n({med:.2f})", fontsize=14, ha="center", va="bottom")
axes[1].text(perc25-0.02, 0.6, f"Perc. 25\n ({perc25:.2f})", fontsize=14, ha="right")
axes[1].text(perc75+0.02, 0.6, f"Perc. 75\n ({perc75:.2f})", fontsize=14, ha="left")

axes[1].spines[["top", "right"]].set_visible(False)
axes[1].spines[:].set_linewidth(2)

axes[1].set_xlabel("Edge weight (Asymmetry)", fontsize=16)
axes[1].set_ylabel("Density", fontsize=16)
axes[1].tick_params(axis="both", which="major", labelsize=14)

axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.1)

all_axes[0].text(-0.2, 1, "A.", fontsize=16, fontweight="bold", transform=all_axes[0].transAxes)
all_axes[0].text(1.2, 1, "B.", fontsize=16, fontweight="bold", transform=all_axes[0].transAxes)

_, _, cbar = plot.add_cbar(fig, ax=all_axes[0])
cbar.ax.set_yticks([0, 0.5, 1], labels=["0", "Bidirectional", "1"], rotation=90, va="center", fontsize=14)

fig.savefig(op.join(path_to_figures, f"SI-DirectedConnectome.png"), dpi=300, bbox_inches='tight')
fig.savefig(op.join(path_to_figures, f"SI-DirectedConnectome.pdf"), dpi=300, bbox_inches='tight', format="pdf")

## SI-EdgeClusters

In [ ]:
hierarchical_matrix = 1 - avg_cons
# hierarchical_matrix = 1 - avg_cons_fix
c_mat = 1 - hierarchical_matrix

dist_condensed = squareform(hierarchical_matrix)
Z = linkage(dist_condensed, method='average', metric='precomputed')
distances = np.flip(Z[:, 2])
diff_dist = -np.diff(distances)

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.signal import argrelextrema

n_samples = len(hierarchical_matrix)

# Number of clusters K decreases from n_samples to 1
n_clusters = np.arange(10, 100, 1)
n_clusters = np.arange(5, 81, 1)

# precompute pair indices for condensed vector
p_idx, q_idx = np.triu_indices(n_samples, k=1)

best_k = np.argmax(diff_dist)+1
print(f"Best k is at max diff: {best_k} with distance jump {diff_dist[best_k-1]:.4f}")

all_ratios = np.zeros(len(n_clusters))
for i, k in enumerate(n_clusters):
    labs = fcluster(Z, t=k, criterion='maxclust')

    pair_same = (labs[p_idx] == labs[q_idx])
    within_val = 1 - dist_condensed[pair_same]
    between_val = 1 - dist_condensed[~pair_same]

    within_mean = within_val[within_val > 0].mean()
    if len(between_val[between_val > 0]) == 0:
        print("Oopsie")
        all_ratios[i] = np.nan
        continue
    else:
        between_mean = between_val[between_val > 0].mean()

    all_ratios[i] = within_mean / between_mean

all_ratios = np.nan_to_num(all_ratios, nan=np.nanmax(all_ratios))

local_max_idx = argrelextrema(diff_dist, np.greater)[0]
ordered_maximas = local_max_idx[np.flip(np.argsort(diff_dist[local_max_idx]))]

print("Top 10 largest jumps in distance:")
best_ks = []
best_cuts = []
for i, top10 in enumerate(ordered_maximas[:10]):
    print(f" -{i+1:02d}: K = {top10+2},\tdist: {diff_dist[top10]:.4f}, \tcut: {distances[top10]:.4f}")
    best_ks.append(top10+2)
    best_cuts.append(distances[top10])

In [ ]:
importlib.reload(dgsp)
importlib.reload(plot)

show_consensus = True
n_samples = len(hierarchical_matrix)
p_idx, q_idx = np.triu_indices(n_samples, k=1)

n_top_k = 10
ncols = n_top_k // 2
fig, all_axes = plt.subplots(nrows=2, ncols=ncols, figsize=(5*ncols, 11))
all_axes = all_axes.flatten()

cut_dist = distances[:-1] - diff_dist/2
selected_cuts = np.flip(sorted(cut_dist))

kept_e_labels = []
for i, axes_bc in enumerate(all_axes):
    dist = selected_cuts[ordered_maximas[i]]
    e_labels = fcluster(Z, t=dist, criterion='distance')
    e_mat = np.zeros_like(graph, dtype=int)
    e_mat[graph != 0] = e_labels

    if i < 3:
        kept_e_labels.append(e_labels)

    cluster_cmap = cmaps["extended_ncar"].resampled(e_labels.max()+1)
    axes_bc.imshow(e_mat[order_by_lobe][:, order_by_lobe], cmap=cluster_cmap, vmin=0, vmax=len(np.unique(e_labels))+1, interpolation="none")
    plot.plot_lobe_lines(axes_bc, lobe_sizes, lobe_labels, no_insula=True, x_hemi=True, fontsize=14)
    
    if i not in [0, ncols]:
        axes_bc.set_yticks([])
    
    title_y = 1.1
    if i >= ncols:
        axes_bc.set_xticks([])
        title_y = 1

    if show_consensus:
        pair_same = (e_labels[p_idx] == e_labels[q_idx])
        within_val = 1 - dist_condensed[pair_same]
        between_val = 1 - dist_condensed[~pair_same]

        within_mean = within_val[within_val > 0].mean()
        if len(between_val[between_val > 0]) == 0:
            print("Oopsie")
            cons_ratio = np.nan
            continue
        else:
            between_mean = between_val[between_val > 0].mean()

        cons_ratio = within_mean / between_mean
        
        axes_bc.set_title(f"$K = {e_mat.max():d}$ Bicommunities\n"+
                          f"Linkage Distance: {diff_dist[ordered_maximas[i]]:.3f}\n"+
                          f"Consensus Ratio: {cons_ratio:.3f}", fontsize=16, y=title_y)
    else:
        axes_bc.set_title(f"$K = {e_mat.max():d}$ Bicommunities\nLinkage Distance: {diff_dist[ordered_maximas[i]]:.3f}", fontsize=16, y=title_y)

consensus_suffix = "_WithConsensus" * show_consensus
fig.savefig(op.join(path_to_figures, f"SI-EdgeClusters{consensus_suffix}.png"), dpi=300, bbox_inches='tight')
fig.savefig(op.join(path_to_figures, f"SI-EdgeClusters{consensus_suffix}.pdf"), dpi=300, bbox_inches='tight', format="pdf")

## SI-DirSignificance

In [ ]:
importlib.reload(dgsp)

compute_all = False

n_shuffle = 10000
print(f"Max p-val: {1 / (1 + n_shuffle)}")
print(f"Max corr p-val (k=80): {80 / (1 + n_shuffle)}")
perm_prop = 0.5

all_shuffled = dgsp.shuffle_edges_sym(graph, perm_prop=perm_prop, n_shuffle=n_shuffle)

shuffled_asym = [np.divide(shuffled, (shuffled + shuffled.T),
                           where=(shuffled + shuffled.T) > 0,
                           out=np.zeros_like(shuffled)) for shuffled in all_shuffled]

all_sig_props = np.zeros(len(n_clusters))

kept_ks = [elab.max() for elab in kept_e_labels]
all_sig_dfs = []

parallel = Parallel(n_jobs=14, return_as="generator")

if compute_all:
    counter = tqdm(total=len(n_clusters)*n_shuffle)
else:
    counter = tqdm(total=len(kept_ks)*n_shuffle)

save_perm = False
# path_to_saveperm = f"./results/BrainKmeans/scale{scale}/permutations"
path_to_saveperm = op.join(path_to_derivatives, "permutations", f"scale{scale}")

for k_i, k in enumerate(n_clusters):
    if (k not in kept_ks) and (not compute_all):
        continue

    e_clust = fcluster(Z, t=k, criterion='maxclust')
    edge_clusters_mat = np.zeros_like(graph, dtype=int)
    edge_clusters_mat[graph != 0] = e_clust

    # bicoms_masks = np.array([edge_clusters_mat == (i+1) for i in range(edge_clusters_mat.max())])
    bicoms_masks = np.array([e_clust == (i+1) for i in range(e_clust.max())])
    # true_ratio = dgsp.get_conjugate_ratio(graph, row_ind=row_ind, col_ind=col_ind, bicom_masks=bicoms_masks)
    graph_asym = np.divide(graph, (graph + graph.T), where=(graph + graph.T) > 0, out=np.zeros_like(graph))
    true_ratio = dgsp.get_asym_ratio(graph_asym[graph > 0], bicom_masks=bicoms_masks)

    fname = f"Dir-permutations_scale{scale}_gamma{gamma}-{n_shuffle}Perm-K{edge_clusters_mat.max()}"
    fname += ".pkl"
    # print(f"Savename is {fname}")

    if op.isfile(op.join(path_to_saveperm, fname)):
        print("Loading existing permutations...")
        perm_data = dload.load(op.join(path_to_saveperm, fname))
        all_shuffled = perm_data["all_shuffled"]
        all_ratios_sym = perm_data["all_ratios_sym"]
    else:
        all_ratios_sym = np.zeros((n_shuffle, len(true_ratio)))
        
        mygen = parallel(delayed(dgsp.get_asym_ratio)(shuffled_asym[s_i][graph > 0], bicom_masks=bicoms_masks) for s_i in range(n_shuffle))
        
        for g_i, res in enumerate(mygen):
            all_ratios_sym[g_i] = res
            counter.update(1)

        all_ratios_sym = np.nan_to_num(all_ratios_sym, nan=0.5)

        if save_perm and (not compute_all):
            dload.save(op.join(path_to_saveperm, fname),
                       {"all_shuffled": all_shuffled,
                        "all_ratios_sym": all_ratios_sym})

    all_ps = np.ones_like(true_ratio)
    for i, t in enumerate(true_ratio):
        # biggersum = np.sum(np.abs(all_ratios_sym[:, i] - all_ratios_sym[:, i].mean()) >= abs(t - all_ratios_sym[:, i].mean()))
        biggersum = np.sum(np.abs(all_ratios_sym[:, i] - 0.5) >= abs(t - 0.5))
        p_two = (1 + biggersum) / (1 + n_shuffle)
        all_ps[i] = p_two

    sig_df = pd.DataFrame({
        "Bicomm_ID": np.arange(1, len(true_ratio)+1),
        "directionality": true_ratio,
        "abs_dir": np.abs(true_ratio - 0.5),
        "p_value": all_ps,
        "p_corr": all_ps * len(all_ps)
        })

    sig_df["is_sig"] = sig_df["p_corr"] < 0.05
    sig_df.sort_values(["is_sig", "abs_dir"], ascending=[False, False], inplace=True)

    all_sig_props[k_i] = sig_df.is_sig.values.sum() / len(sig_df)

    if k in kept_ks:
        all_sig_dfs.append(sig_df)

counter.close()

In [ ]:
from matplotlib import patheffects as PathEffects

if compute_all:
    fig, axes = plt.subplots(nrows=3, figsize=(12, 8), gridspec_kw={"hspace": 0.2}, sharex=True)

    axes[0].plot(n_clusters, diff_dist[n_clusters[0]-2:n_clusters[-1]-1], color="grey", lw=2, marker='o', markersize=4, zorder=2)
    axes[1].plot(n_clusters, all_ratios, color="grey", lw=2, marker='o', markersize=4, zorder=2)
    axes[2].plot(n_clusters, all_sig_props, color="grey", lw=2, marker='o', markersize=4, zorder=2)

    legend_handles = [
        Line2D([0], [0], color='tab:red', lw=4, label='Top 3', zorder=1),
        Line2D([0], [0], color='tab:blue', lw=4, alpha=0.8, label='Top 10', zorder=2)
    ]
    for ax in axes:
        for kept_k in ordered_maximas[:10]:
            ax.axvline(kept_k + 2, color='tab:blue', lw=3, zorder=0, alpha=0.8)

        for kept_k in kept_ks:
            ax.axvline(kept_k, color='tab:red', lw=3, zorder=1)
            
            txt = ax.text(kept_k+0.5, ax.get_ylim()[1]*0.9, f"$K={kept_k}$", fontsize=12)
            txt.set_path_effects([PathEffects.withStroke(linewidth=2, foreground='white', alpha=0.8)])


        ax.tick_params(labelsize=12)
        ax.legend(handles=legend_handles, fontsize=14)

        # ax.set_xlim(5.5, 69.5)

    axes[2].set_xlabel("Number of Bicommunities $(K)$", fontsize=16)
    axes[0].set_title("Linkage Distance", fontsize=14)
    axes[0].set_ylabel("Distance", fontsize=14)
    axes[1].set_title("Consensus Stability", fontsize=14)
    axes[1].set_ylabel("Consensus Ratio", fontsize=14)
    axes[2].set_title("Significantly Asymmetric Bicommunities", fontsize=14)
    axes[2].set_ylabel("Proportion of\nBicommunities", fontsize=14)

    fig.savefig(op.join(path_to_figures, f"SI-DirSignificance.png"), dpi=300, bbox_inches='tight')
    fig.savefig(op.join(path_to_figures, f"SI-DirSignificance.pdf"), dpi=300, bbox_inches='tight', format="pdf")

## SI-BundleOverlap

In [ ]:
n_yeo = 7
# n_yeo = 17

path_to_yeo_corr = op.join(path_to_derivatives, "atlas_correspondence", "YeoNetworks")
corr_fname = f"Laus2018_Yeo{n_yeo}-scale{scale}OneThal.pkl"

yeo_corr = dload.load(os.path.join(path_to_yeo_corr, corr_fname))

corr_mat = yeo_corr["dice"]
yeo_labels = yeo_corr["labels"]

if n_yeo == 7:
    yeo_colors_dict = {
    "Visual": "#781286",
    "Somatomotor": "#4682B4",
    "Dorsal Attention": "#00760E",
    "Ventral Attention": "#C43AFA",
    "Limbic": "#DC6D04",
    "Frontoparietal": "#E69422",
    "Default": "#CD3E4E",
    "Subcortical": "#6D0500",
    }
    yeo_colors = [yeo_colors_dict[label] for label in yeo_labels]

else:
    yeo_colors = [
    "#781286", "#9B4F96",
    "#4682B4", "#5AA0D6",
    "#00760E", "#4FAF4A",
    "#C43AFA", "#E89CF4",
    "#DC6D04", "#F2A541",
    "#E69422", "#F6C85F", "#FFD92F",
    "#CD3E4E", "#E57373", "#F28E8E",
    "#8E2F2F", "#6D0500",
    ]

# Abrev. of networks
yeo_labels[2] = "Dorsal Attn."
yeo_labels[3] = "Ventral Attn."


s_e_labels = kept_e_labels[0]
sig_df = all_sig_dfs[2]

print(s_e_labels.max(), "edge bicommunities found at K =", kept_ks[1])
print(len(sig_df), "bicomm significance assessed")

if len(sig_df) != s_e_labels.max():
    raise ValueError("Warning: mismatch between sig_df and s_e_labels")

s_e_mat = np.zeros_like(edge_clusters_mat)
s_e_mat[edge_clusters_mat > 0] = s_e_labels

cluster_cmap = cmaps["extended_ncar"].resampled(s_e_mat.max()+1)

In [ ]:
from matplotlib.patches import Rectangle

e_lab_one_hot = np.array([s_e_labels == k+1 for k in range(s_e_labels.max())]).astype(float)
e_lab_one_hot = e_lab_one_hot / e_lab_one_hot.sum(axis=1, keepdims=True)
e2b = edge_to_bundle_dist / edge_to_bundle_dist.sum(axis=1, keepdims=True)

bicom_to_bundle = e_lab_one_hot @ e2b

fig, axes_barplot = plt.subplots(figsize=(15, 8))

e2b_max = e2b.max(axis=0)
b2b_max = bicom_to_bundle.max(axis=0)

sort_by_overlap = np.argsort(b2b_max)[::-1]

perc_thresh = 90
thresh = np.percentile(e2b, perc_thresh, axis=0)
overlap_mask = bicom_to_bundle >= thresh

n_a_show = 51

jitter_width = 0.4
for i, a_id in enumerate(sort_by_overlap[:n_a_show]):
    x_scatter = np.random.uniform(-jitter_width, jitter_width, size=e2b[:, a_id].shape[0])
    axes_barplot.scatter(i + x_scatter, e2b[:, a_id], color="gray", alpha=0.1, s=5, edgecolors="none")
    axes_barplot.plot([i - jitter_width, i + jitter_width], [thresh[a_id]]*2, color="k", lw=2, zorder=5)
    
    x_scatter = np.random.uniform(-jitter_width, jitter_width, size=bicom_to_bundle[:, a_id].shape[0])
    
    sort_bicom = np.flip(np.argsort(bicom_to_bundle[:, a_id]))
    sort_bicom = sort_bicom[:3]
    full_width = 0.7
    width = full_width/sort_bicom.size
    axes_barplot.bar(i - np.linspace(0.42-width/2, width/2-0.42, sort_bicom.size),
                     bicom_to_bundle[sort_bicom, a_id], width=width, color=cluster_cmap(sort_bicom), alpha=0.6, zorder=3)
    axes_barplot.bar(i - np.linspace(0.42-width/2, width/2-0.42, sort_bicom.size),
                     bicom_to_bundle[sort_bicom, a_id], width=width, color="none", edgecolor=cluster_cmap(sort_bicom), lw=1, zorder=4)

legend_handles = [Line2D([0], [0], marker='o', color='w', label=f'Edge Similarity', markerfacecolor='gray', markersize=5),
                  Line2D([0], [0], color='k', lw=3, label=f'Edge {perc_thresh}th Percentile Threshold'),
                  Rectangle((0, 0), 1, 1, facecolor=cluster_cmap(6), edgecolor=cluster_cmap(6), lw=1, alpha=0.6, label='Bicommunity Similarity (top 3)')]

axes_barplot.legend(handles=legend_handles, fontsize=14, loc='upper right')

axes_barplot.set_xticks(np.arange(n_a_show), labels=a_bundles_labels["BundleNameShort"][sort_by_overlap[:n_a_show]],
                        rotation=90, fontsize=10, ha="center")

axes_barplot.set_ylim(0, 1.2*b2b_max.max())
axes_barplot.set_xlim(-1, n_a_show)
axes_barplot.set_ylabel("Inverse MDF Distance", fontsize=16)
axes_barplot.set_title("Similarity with Anatomical Bundles", fontsize=16)
axes_barplot.tick_params(labelsize=14, width=2)

axes_barplot.spines[["top", "right"]].set_visible(False)
axes_barplot.spines[:].set_linewidth(2)

fig.savefig(op.join(path_to_figures, f"SI-BundleOverlap-K{s_e_labels.max()}-p{perc_thresh}.png"), dpi=300, bbox_inches='tight')
fig.savefig(op.join(path_to_figures, f"SI-BundleOverlap-K{s_e_labels.max()}-p{perc_thresh}.pdf"), dpi=300, bbox_inches='tight', format="pdf")

## SI-AnatomicalBicom

In [ ]:
importlib.reload(plot)
path_to_centroids = op.join(path_to_bundles, "atlas", "pop_average")

label_space = 1.2
width_mod = 4
maxoverlap = 4
surf_overlay = 0.1
node_alpha = True

ncols = 5
nrows = np.ceil(s_e_labels.max() / ncols).astype(int)
n_nodes = graph.shape[0]

fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(10*ncols, 3*nrows),
                         gridspec_kw={'wspace':0.02, 'hspace':0})
axes = axes.flatten()

for ax in axes:
    ax.axis("off")

maxoverlaps = [maxoverlap]*s_e_labels.max()

manual_ab_names = {
    'AC': "Anterior Commissure",
    'AF': "Arcuate Fasciculus",
    'CC_Oc': "Corpus Callosum Occipital",
    'CC_Te': "Corpus Callosum Temporal",
    'CC_Pa': "Corpus Callosum Parietal",
    'CC_Fr_1': "Corpus Callosum Front. 1",
    'CC_Fr_2': "Corpus Callosum Front. 2",
    'CC_Pr_Po': "Corpus Callosum Post.",
    'CG': "Cingulum",
    'CG_An': "Cingulum Ant.",
    'CG_Po': "Cingulum Post.",
    'CG_curve': "Cingulum Curve",
    'FAT': "Front. Aslant Tract",
    'FPT': "Fronto-pontine tract",
    'FPT_Brainstem': "Fronto-pontine & Brainstem",
    'FX': "Fornix",
    'ILF': "Inf. Long. Fasciculus",
    'MdLF': "Middle Long. Fasciculus",
    'OR_ML': "Optic Radiation",
    'PYT': "Pyramidal tract",
    'PYT_Brainstem': "Pyramidal & Brainstem",
    'SLF': "Sup. Long. Fasciculus",
    'UF': "Uncinate Fasciculus"
    }

n_abundle = len(manual_ab_names)
abundle_cmap = cluster_cmap.resampled(n_abundle + 1)
all_ab_names = {}
for ax_i in np.arange(s_e_labels.max()):
    # if ax_i > 1:
    # if ax_i < s_e_labels.max() - 3:
    #     continue
    # for i, a_id in enumerate(sort_by_overlap[:n_a_show]):

    bic_id = ax_i
    best_ab = np.flip(np.argsort(bicom_to_bundle[bic_id]))
    
    ab_names = []

    last_overlap = 0
    n_overlap = 0
    for ab_i, ab in enumerate(best_ab):
        ab_name = a_bundles_labels.loc[ab, "BundleName"]

        if last_overlap == 0:
            last_overlap = bicom_to_bundle[bic_id, ab]

        if (bicom_to_bundle[bic_id, ab] < 0.75 * last_overlap) or (bicom_to_bundle[bic_id, ab] < 0.5 * bicom_to_bundle[:, ab].max()) or (n_overlap >= maxoverlaps[ax_i]):
            break
        else:
            last_overlap = bicom_to_bundle[bic_id, ab]
            ab_names.append(ab_name)
            n_overlap += 1

            ab_name_nolat = ab_name.replace("_L", "").replace("_R", "")
            if ab_name_nolat not in all_ab_names.keys():
                all_ab_names.update({ab_name_nolat: len(all_ab_names.keys()) + 1})
    
    stripednames = [name.replace("_L", "").replace("_R", "") for name in ab_names]
    color_idxs = [all_ab_names[name] for name in stripednames]

    colors = np.array([abundle_cmap(idx) for idx in color_idxs])
    path_to_trk = [op.join(path_to_centroids, f"{name.replace(' ', '_')}.trk") for name in ab_names]

    is_sig = sig_df.loc[sig_df['Bicomm_ID'] == bic_id + 1, 'is_sig'].values[0]
    asymmetry = sig_df.loc[sig_df['Bicomm_ID'] == bic_id + 1, 'directionality'].values[0]

    e_clust_mask = np.zeros_like(s_e_mat)
    e_clust_mask[s_e_mat == bic_id + 1] = 1

    subnet_mask = corr_mat @ e_clust_mask @ corr_mat.T

    asymmetry = sig_df.loc[sig_df['Bicomm_ID'] == bic_id + 1, 'directionality'].values[0]

    if asymmetry < 0.5:
        subnet_mask = subnet_mask.T
        e_clust_mask = e_clust_mask.T

    is_sig = sig_df.loc[sig_df['Bicomm_ID'] == bic_id + 1, 'is_sig'].values[0]

    if not is_sig:
        e_clust_mask += e_clust_mask.T
        e_clust_mask[e_clust_mask > 0] = 1

        subnet_mask = (subnet_mask + subnet_mask.T)/2

    if len(ab_names) > 4:
        ab_name = ", ".join(ab_names[:4]) + "\n" + ", ".join(ab_names[4:]) 
    else:
        ab_name = " , ".join(ab_names)

    # title = f"#{bic_id + 1} - {ab_name}" + (" $(*)$" if is_sig else "")
    # axes[ax_i].set_title(title, fontsize=20, zorder=3, y=1)
    axes[ax_i].axis("off")
    
    gs = GridSpecFromSubplotSpec(1, 3, axes[ax_i].get_subplotspec(), width_ratios=[1, 1, 1.1], wspace=0)
    sub_axes = [fig.add_subplot(gs[i]) for i in range(3)]
    
    # sub_axes[0].text(0, 0.9, f"{ascii_uppercase[ax_i]}.", fontsize=16, transform=sub_axes[0].transAxes)
    sub_axes[0].text(0, 0.9, f"#{ax_i+1:d}", fontsize=16, transform=sub_axes[0].transAxes)

    RR = e_clust_mask[:n_nodes//2][:, :n_nodes//2]
    LL = e_clust_mask[n_nodes//2:-1][:, n_nodes//2:-1]
    LR = e_clust_mask[:n_nodes//2][:, n_nodes//2:-1] 
    RL = e_clust_mask[n_nodes//2:-1][:, :n_nodes//2]

    between_hemi = (LR + RL).sum() > (LL + RR).sum()
    if between_hemi:
        myview = "tra"
        # if e_clust_mask[:, -1].sum() + e_clust_mask[-1].sum() > 0:
        #     myview = "back"
    else:
        if LL.sum() > RR.sum():
            myview = "left"
        else:
            myview = "right"

    # if ax_i > s_e_labels.max() - 2:
    if not fast:
        # Tracks
        plot.plot_trk(path_to_trk, cmap=cmaps["div_rb"], upsample=None, n_slines=500,
                    brain_opacity=surf_overlay, overlay_slines=surf_overlay < 1,
                    trk_color_list=colors,
                    view=myview, linewidth=0.3, slines_alpha=1, axes=sub_axes[0])

        # Bicom
        axpos = sub_axes[1].get_position()
        cbar_pos = np.array(axpos.bounds)
        cbar_pos[0] += cbar_pos[2] * 0.1
        cbar_pos[2] *= 0.8
        cbar_pos[1] += cbar_pos[3] * 0.2
        cbar_pos[3] = 0.01
        plot.plot_bicom_tracts(1, e_clust_mask, labels, scale=2, cmap=cmaps["div_rb"], n_centroids=5,
                            brain_opacity=surf_overlay, overlay_slines=surf_overlay < 1,
                            linewidth=0.3, slines_alpha=1 - 0.01 * is_sig, bidir_col=(0.6, 0.6, 0.6, 1),
                            view=myview, axes=sub_axes[1], upsample=upsample,
                            fig=fig, plot_cbar=False, cbar_pos=cbar_pos, cbar_fontsize=14)
    # Networks
    sub_axes[2].axis("off")
    sub_axes[2] = plot.plot_yeo_summary(subnet_mask, yeo_labels, cmap=cmaps["div_rb_silver"], axes=sub_axes[2],
                                        width_mod=1, r_lab_only=True, manual_arrows=True, net_alpha=node_alpha)

    if ax_i < 3:
        sub_axes[0].set_title("Anatomical\nBundles", fontsize=14)
        sub_axes[1].set_title("Bicommunity\nStreamlines", fontsize=14)
        sub_axes[2].set_title("Network\nRepresentation", fontsize=14)

ax_legend = fig.add_subplot([0.12, 0.06, 0.78, 0.5], facecolor='none')
ax_legend.axis("off")

label_names = [manual_ab_names[name] for name, num in all_ab_names.items()]
legend_handles = [
    # Line2D([0], [0], color=abundle_cmap(num), lw=4, label=f"{name}") for name, num in all_ab_names.items()
    Line2D([0], [0], color=abundle_cmap(num), lw=4, label=f"{label_names[ai]}") for ai, (name, num) in enumerate(all_ab_names.items())
]
# ax_legend.legend(handles=legend_handles, fontsize=12, loc='lower left', ncols=3, title="Anatomical Bundles", title_fontsize=16)
ax_legend.legend(handles=legend_handles, fontsize=12, loc='lower left', ncols=8, title="Anatomical Bundles", title_fontsize=16)

# cb = plot.draw_cbar(fig, cmap=cmaps["div_rb"], ax_pos=[0.4, 0.03, 0.22, 0.05], fontsize=12)
cb = plot.draw_cbar(fig, cmap=cmaps["div_rb"].reversed(), ax_pos=[0.625, 0.076, 0.25, 0.02], fontsize=12,
                    ticks=[0, 10], labels=["Sending", "Receiving"])
cb.set_label("Bicommunity Streamlines and Networks", fontsize=16, labelpad=-70)

if not fast:
    overlay_suffix = f"-overlay{surf_overlay}" if surf_overlay < 1 else ""
    # alpha_suffix = f"-NAlpha" if node_alpha else ""
    fig.savefig(op.join(path_to_figures, f"SI-AnatomicalBicom-WIDE-K{s_e_labels.max()}-max{maxoverlap}{overlay_suffix}.png"), dpi=300, bbox_inches='tight')
    fig.savefig(op.join(path_to_figures, f"SI-AnatomicalBicom-WIDE-K{s_e_labels.max()}-max{maxoverlap}{overlay_suffix}.pdf"), dpi=300, bbox_inches='tight', format="pdf")

# SI-BicomOverlap

In [ ]:
e2b_max = e2b.max(axis=0)

bicom_to_bundle = e_lab_one_hot @ e2b
b2b_max = bicom_to_bundle.max(axis=0)
b2b_max = bicom_to_bundle.max(axis=1)

sort_by_overlap = np.argsort(b2b_max)[::-1]

perc_thresh = 90
thresh = np.percentile(e2b, perc_thresh, axis=0)
overlap_mask = bicom_to_bundle >= thresh

n_a_show = 51
fig, axes = plt.subplots(figsize=(18, 6))

jitter_width = 0.4
for i, b_id in enumerate(sort_by_overlap):
    a_id = np.argmax(bicom_to_bundle[b_id])
    axes.bar(i, b2b_max[b_id], width=0.6, color=cluster_cmap(b_id+1), alpha=0.6, zorder=2)
    axes.bar(i, b2b_max[b_id], width=0.6, color="none", edgecolor=cluster_cmap(b_id+1), lw=3, zorder=2)
    
    x_scatter = np.random.uniform(-jitter_width, jitter_width, size=e2b[:, a_id].shape[0])
    axes.scatter(i + x_scatter, e2b[:, a_id], color="gray", alpha=0.1, s=10, edgecolors="none")
    axes.plot([i - jitter_width, i + jitter_width], [thresh[a_id]]*2, color="k", lw=3, zorder=2)

    axes.text(i, b2b_max.max() + 0.028, a_bundles_labels.loc[a_id, "BundleName"], color="k", fontsize=12, ha="center", va="bottom", rotation=90)
    
    perc = 100- (1 + (e2b[:, a_id] >= b2b_max[b_id]).sum())/(1 + e2b.shape[0]) * 100
    axes.text(i, 0.07, f"{perc:2.1f}", color="k", fontsize=12, ha="center", va="top", rotation=90)
    
    # bicom_list = [9, 24, 1, 16, 0, 12, 11, 23]
    # if b_id in [9, 24, 1, 16, 0, 12, 11, 23]:
    if b_id in [11, 29, 4, 26, 13, 33, 2, 12]:
        axes.scatter(i, 0.075, s=80, marker="s", facecolors='r', edgecolors="none", zorder=3)
    
    if sig_df.loc[b_id, 'is_sig']:
        axes.scatter(i, 0.075, s=80, marker="s", facecolors='none', edgecolors="k", linewidths=1, zorder=3)
    # x_scatter = np.random.uniform(-jitter_width, jitter_width, size=bicom_to_bundle[:, a_id].shape[0])
    # axes.scatter(i + x_scatter, bicom_to_bundle[:, a_id], c=np.arange(bicom_to_bundle[:, a_id].shape[0]),
    #              cmap=cluster_cmap, alpha=1, s=50, edgecolors="none")

axes.text(-1, b2b_max.max() + 0.036, "Bundle\nName", fontsize=12, rotation=90, ha="center", va="center")
axes.text(-1, 0.065, "Percentile", fontsize=12, rotation=90, ha="center", va="center")

axes.set_xticks(np.arange(len(sort_by_overlap)), labels=sort_by_overlap+1, rotation=0, fontsize=10)
axes.axhline(0, color='k', lw=2)

axes.spines[["top", "right"]].set_visible(False)
axes.spines[:].set_linewidth(2)
axes.set_xlabel("Bicommunity ID", fontsize=16)
axes.set_ylabel("Inverse MDF Distance", fontsize=16)
axes.set_title("Bicommunity Similarity with Anatomical Bundles", fontsize=16)
axes.tick_params(labelsize=14, width=2)

fig.savefig(op.join(path_to_figures, f"SI-BicomOverlap-K{s_e_labels.max()}.png"), dpi=300, bbox_inches='tight')
fig.savefig(op.join(path_to_figures, f"SI-BicomOverlap-K{s_e_labels.max()}.pdf"), dpi=300, bbox_inches='tight', format="pdf")